## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 28 samples and 15003 columns.
First few rows of the data:


,RFS (months),RFS Censor,Patient ID,RNA-Seq ID,Date of Esophageal Surgery,Relapse,Date of Relapse or Censoring,Date of Death,B cells naive,B cells memory,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,26.1184,0,B012-0001,SL462108,2016-04-01,No,2018-06-04,NaN,0.022107,0.00000,...,3.662069,2.487979,3.810792,3.733372,1.791442,2.494363,4.520041,4.071395,2.996526,3.639040
1,24.5724,0,B012-0004,SL462135,2016-06-10,No,2018-06-27,NaN,0.019443,0.00000,...,2.473485,1.498157,2.663531,2.518551,1.111765,1.356405,3.285345,3.232526,2.252723,2.644813
2,14.0461,1,B012-0007,SL446519,2016-08-01,Yes,2017-10-02,NaN,0.037155,0.00000,...,4.116829,3.025456,4.343368,4.213837,2.025170,2.679345,4.979327,4.475582,3.544341,4.142523
3,17.1382,0,B012-0008,SL462109,2016-10-14,No,2018-03-19,NaN,0.010599,0.00000,...,2.842831,1.888814,3.229611,3.045209,1.598104,2.053633,3.996419,3.611956,2.599311,3.154038
4,24.9671,1,B012-0011,SL446521,2016-12-19,No,2019-01-17,2019-01-17,0.030652,0.00247,...,2.953496,2.304290,3.511578,3.084574,1.602120,2.004306,4.262906,4.016896,3.214875,3.300263


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
